# Retriever :
```txt
[User Query] 
      ↓
[Retriever] ──→ (Searches Vector Database / Document Store)
      ↓
[Relevant Chunks Found]
      ↓
[Augmented Prompt] (Query + Chunks)
      ↓
[LLM (Generator)]
      ↓
[Final Answer]
```
# Lesson: Understanding Retrievers in RAG

## Introduction
Retrieval-Augmented Generation (RAG) is a technique used to grant Large Language Models (LLMs) access to data they weren't originally trained on. The **Retriever** is the engine that fetches this data.

## Core Concepts

### 1. The "Vector" Magic
Most modern retrievers use **Embeddings**. 
* **Text:** "I love cats."
* **Embedding:** `[0.12, -0.59, 0.88, ...]`
By turning text into coordinates in a multi-dimensional space, the retriever finds documents that are "close" to the query.

### 2. Common Retrieval Strategies
* **Similarity Search:** The standard "find things that look like this" approach.
* **Max Marginal Relevance (MMR):** Finds relevant items but ensures they are *diverse* so the LLM doesn't get 5 copies of the same sentence.
* **Re-ranking:** The retriever finds 20 results, and a second "Ranker" model picks the best 5.

## Why do we need them?
1. **Reduce Hallucination:** The model cites actual sources.
2. **Up-to-date Info:** You can update your database daily without retraining the LLM.
3. **Data Privacy:** You can store your private company files in a retriever without sending them to a public model's training set.

## Summary Checklist
- [ ] User asks a question.
- [ ] Retriever searches the Vector DB.
- [ ] Top-K (best) results are pulled.
- [ ] Results are sent to the LLM as context.

```txt
graph TD
    A[User Query] --> B{Router / Agent}
    B -- "I need tech facts" --> C[Wiki Retriever]
    B -- "I need research papers" --> D[ArXiv Retriever]
    B -- "I need company data" --> E[Vector Store]
    C --> F[Relevant Context]
    D --> F
    E --> F
    F --> G[LLM Generator]
    G --> H[Final Answer]
```
# Lesson 2: Source-Specific & Hybrid Retrievers

## 1. Beyond the Vector Store
While Vector Stores (like Pinecone or Chroma) are popular, RAG can use any "Retriever" as long as it returns text.

### The "Wiki" Retriever
* **Source:** Wikipedia API.
* **Benefit:** Zero storage cost. You don't need to save the data; you just fetch it live.
* **Use Case:** General knowledge bots.

### The "ArXiv" Retriever
* **Source:** Scientific paper repositories.
* **Benefit:** Provides highly specialized, peer-reviewed data.
* **Use Case:** Scientific research assistants.

## 2. Hybrid Search (The Industry Standard)
Most professional systems don't rely on just one. They use **Hybrid Retrieval**:
1.  **Dense Retrieval (Vector):** Finds "The capital of France" if you ask "Where does the French government sit?"
2.  **Sparse Retrieval (Keyword):** Finds "Model-X12-Blue" if the user types that exact part number.

## 3. Key Terms for Your Exams/Projects
* **Top-K:** The number of documents the retriever grabs (usually 3 to 5).
* **Score Threshold:** A filter to ignore results that aren't relevant enough.
* **Router:** An LLM "manager" that decides which retriever (Wiki, ArXiv, or Vector) to use for a specific question.

In [1]:
# Wikipedia Retriever

from langchain_community.retrievers.wikipedia import WikipediaRetriever
#initiate retriever with default settings with toped 5 results  
retriever = WikipediaRetriever(top_k_results=5, lang="en")  
#Define your query 
query="The Geopolitical History of India and Pakistan from perspective of Kashmir"
#get the relevent wikipedia articles for the query
results = retriever.invoke(query)
print(results)

[Document(metadata={'title': 'India–Pakistan war of 1971', 'summary': "The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.\nThirteen days after the war started, India achieved a clear upper hand, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as th

In [2]:
for i, doc in enumerate(results):
    print(f"================ Result {i+1} ================")
    print("Title:", doc.metadata.get("title"))
    print("Source:", doc.metadata.get("source"))
    print("Content Preview:", doc.page_content[:300])
    print("\n")

================ Result 1 ================
Title: India–Pakistan war of 1971
Source: https://en.wikipedia.org/wiki/India%E2%80%93Pakistan_war_of_1971
Content Preview: The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began w


================ Result 2 ================
Title: India–Pakistan war of 1965
Source: https://en.wikipedia.org/wiki/India%E2%80%93Pakistan_war_of_1965
Content Preview: The India–Pakistan war of 1965, also known as the second India–Pakistan war, was an armed conflict between Pakistan and India that took place from August 1965 to September 1965.
The conflict began following Pakistan's unsuccessful Operation Gibraltar, which was designed to infiltrate forces into Jam


================ Result 3 ================
Title: India–Paki

# Vector Store Retriever :
We will use the knowledge Base (Vector Database) as source we will retrieve documents from DB.


In [3]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/Users/vinod/DaasAI/GenAI_pract/venv_cromadbsetup/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]
# Create a Chroma vector store

vectorestore = Chroma.from_documents(documents=documents,
                                      embedding=embeddings,
                                      collection_name="Langchain_documents")


In [5]:
# Convert vector Store into retriever 
retriever = vectorestore.as_retriever(search_kwargs={"k": 4})
# Define a query
query = "What helps developers build LLM applications?"
# Retrieve relevant documents
results=retriever.invoke(query)
for i, doc in enumerate(results):
    print(f"================ Result {i+1} ================")
    print("Content:", doc.page_content)
    print("\n")

================ Result 1 ================
Content: LangChain helps developers build LLM applications easily.


================ Result 2 ================
Content: Chroma is a vector database optimized for LLM-based search.


================ Result 3 ================
Content: OpenAI provides powerful embedding models.


================ Result 4 ================
Content: Embeddings convert text into high-dimensional vectors.




In [6]:
results=vectorestore.similarity_search(query, k=4)
for i, doc in enumerate(results):
    print(f"================ Result {i+1} of siilarity method ================")
    print("Content:", doc.page_content)
    print("\n")

================ Result 1 of siilarity method ================
Content: LangChain helps developers build LLM applications easily.


================ Result 2 of siilarity method ================
Content: Chroma is a vector database optimized for LLM-based search.


================ Result 3 of siilarity method ================
Content: OpenAI provides powerful embedding models.


================ Result 4 of siilarity method ================
Content: Embeddings convert text into high-dimensional vectors.




# Advanced RAG Retrieval Strategies: MMR, MQR, and CCR

This document outlines three sophisticated retrieval methods used in Retrieval-Augmented Generation (RAG) to improve the quality, diversity, and precision of AI responses.

---

## 1. Maximal Marginal Relevance (MMR)
""" How can we pick the result that are not only relevent to the query but also different from each other."""

**The Problem:** Standard semantic searches often return documents that are very similar to each other. If the top 5 results all say the same thing in different ways, the LLM loses context it could have gained from other sources.

**The Solution:** MMR re-ranks search results to balance **Relevance** (how well it matches the query) with **Diversity** (how different it is from already selected results).

### How it Works
1. It identifies a large pool of candidate documents via standard vector search.
2. It selects the single most relevant document.
3. For the next selection, it penalizes documents that are too similar to the one(s) already chosen.
4. This iterative process continues until the desired number of diverse documents is reached.



In [7]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [8]:
from langchain_community.vectorstores import FAISS
# Create a FAISS vector store
faiss_vectorestore = FAISS.from_documents(documents=docs, embedding=embeddings)

In [9]:
# Enable  MMR in the retriever 
retriever = faiss_vectorestore.as_retriever(search_kwargs={"k": 4, "lambda_mult": 0.5}, # k = top results, lambda_mult = relevance-diversity balance
                                            search_type="mmr")

In [10]:
query = "What is langchain?"
results = retriever.invoke(query)
for i , doc in enumerate(results):
    print(f"================ Result {i+1} of MMR method ================")
    print("Content:", doc.page_content)
    print("\n")

================ Result 1 of MMR method ================
Content: LangChain supports Chroma, FAISS, Pinecone, and more.


================ Result 2 of MMR method ================
Content: LangChain is used to build LLM based applications.


================ Result 3 of MMR method ================
Content: Embeddings are vector representations of text.


================ Result 4 of MMR method ================
Content: MMR helps you get diverse results when doing similarity search.




## 2. Multi-Query Retrieval (MQR)

**The Problem:** Vector search is highly sensitive to the specific wording of a user's prompt. If a user uses a synonym the database doesn't prioritize, or writes a vague query, the "perfect" document might be missed.

**The Solution:** MQR uses an LLM as a "query expander" to look at the problem from multiple angles.

### How it Works
1. **Query Expansion:** An LLM generates 3–5 variations of the original user query (e.g., "How do I fix a leak?" becomes "Leaking pipe repair," "Plumbing troubleshooting," and "Water damage prevention").
2. **Parallel Search:** The system runs a vector search for *every* version of the query.
3. **Union:** It combines all unique results into one set, ensuring higher "Recall" (the ability to find all relevant info).



In [11]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [17]:
# model calling through Huggingfacehiub
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
import os
from dotenv import load_dotenv
load_dotenv()
model =ChatHuggingFace(llm=HuggingFaceEndpoint(
        repo_id="openai/gpt-oss-20b",
        task="text-generation",
        huggingfacehub_api_token=os.getenv("HUGGINFACE_API_KEY")))
# Create a MultiQueryRetriever with the documents
response = model.invoke("What are the health benefits of regular walking?")
print("Model Response:", response)

Model Response: content='### The Health Benefits of Regular Walking – A Quick‑Reference Guide\n\n| Category | What Happens | Why It Matters | Typical “dose” (WHO/CDC) |\n|----------|--------------|----------------|---------------------------|\n| **Cardiovascular** | Low‑impact aerobic activity that increases heart rate and circulation. | • Reduces risk of heart attack, stroke, and hypertension.<br>• Improves HDL (“good”) cholesterol, lowers LDL (“bad”) and triglycerides. | 150\u202fmin/week of moderate‑intensity walking (e.g., brisk walk). |\n| **Weight Management** | Burns calories and boosts resting metabolic rate. | • Helps maintain or lose weight, especially when paired with healthy eating.<br>• Improves insulin sensitivity, lowering risk of type\u202f2 diabetes. | 30\u202fmin/day, most days of the week. |\n| **Bone & Muscle** | Weight‑bearing movement that stresses joints, bones, and connective tissue. | • Increases bone mineral density, reducing osteoporosis risk.<br>• Strengthen

In [22]:
from langchain.retrievers import MultiQueryRetriever

# Create a MultiQueryRetriever with the documents
vector_store=FAISS.from_documents(all_docs, embedding=embeddings)

#create retriever
siilarity_retriever = vector_store.as_retriever(search_kwargs={"k": 3},search_type="similarity")

#MQR retriver 
mqr_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={"k": 3}),
        llm=model)



ModuleNotFoundError: No module named 'langchain.retrievers'

## 3. Contextual Compression Retriever (CCR)

**The Problem:** Documents are often stored in large "chunks" (e.g., 500–1000 tokens). However, the actual answer might be just one sentence hidden in the middle. Sending massive, noisy chunks to an LLM is expensive and can lead to hallucinations.

**The Solution:** CCR "squeezes" the retrieved documents so only the gold—the actual relevant text—remains.

### How it Works
1. **Base Retrieval:** Standard search pulls back a few large, potentially relevant documents.
2. **Compression:** A "Compressor" (usually a smaller LLM or a Cross-Encoder) scans the documents specifically in the context of the user's question.
3. **Extraction:** It discards irrelevant paragraphs and only passes the specific, pertinent snippets to the final LLM.



---

## Comparison Table

| Strategy | Primary Goal | When to Use |
| :--- | :--- | :--- |
| **MMR** | **Diversity** | When you want to avoid redundancy and get a broad perspective. |
| **MQR** | **Coverage** | When user queries are short, vague, or poorly phrased. |
| **CCR** | **Precision** | When your documents are long/noisy and you need to save tokens. |

---